# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第10章 統計的推論・回帰分析・混合効果モデル ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `sim_measures.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

scripts/R/10_mixed_models.R が読み込む data/processed/measures.csv を
シミュレートする。**実測データではない。** 話者20名×項目30語×条件2水準
(例: 通常発話 vs. 明瞭発話)の交差デザインを想定し、条件の主効果・
対数頻度の主効果・条件と頻度の交互作用を、話者・項目のランダム切片・
ランダム傾きとともに設定する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)
library(tidyverse)

n_speakers <- 20
n_items    <- 30

speakers <- tibble(
  speaker    = sprintf("sp%02d", 1:n_speakers),
  spk_int_re = rnorm(n_speakers, 0, 18),
  spk_slope_re = rnorm(n_speakers, 0, 6)
)
items <- tibble(
  item        = sprintf("item%02d", 1:n_items),
  frequency   = round(exp(runif(n_items, 0, 6))),
  item_int_re = rnorm(n_items, 0, 14)
)

true_intercept   <- 230   # 基準条件・平均頻度での持続時間(ms)
true_cond_effect <- 22    # 条件の主効果(和対比、片側の効果量)
true_freq_effect <- -9    # 対数頻度1単位あたりの持続時間の変化
true_interaction <- 5     # 条件×対数頻度の交互作用

d <- speakers |>
  tidyr::crossing(items) |>
  tidyr::crossing(condition = c("通常", "明瞭")) |>
  mutate(
    logfreq_c = as.numeric(scale(log(frequency), scale = FALSE)),
    cond_c    = if_else(condition == "明瞭", 0.5, -0.5),
    duration_mean = true_intercept +
      true_cond_effect * cond_c +
      (true_freq_effect + spk_slope_re) * logfreq_c +
      true_interaction * cond_c * logfreq_c +
      spk_int_re + item_int_re,
    duration = pmax(duration_mean + rnorm(n(), 0, 20), 40)
  ) |>
  select(speaker, item, condition, frequency, duration)

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
write_csv(d, here::here("data", "processed", "measures.csv"))
cat("生成した行数:", nrow(d), " / 話者数:", n_speakers, " / 項目数:", n_items, "\n")


## `10_mixed_models.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

なぜ混合効果モデルを使うのか:
  音声データは、同じ話者が複数の項目を、同じ項目を複数の話者が産出する
  という交差構造を持つ。話者ごと・項目ごとの平均差を無視して回帰すると、
  独立性の仮定が壊れ、第一種の過誤が膨らむ。ランダム効果は
  「この話者はもともと速い」「この語はもともと長い」を明示的にモデルに
  入れることで、固定効果の推定を話者・項目の偶然から切り離す。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

d <- read_csv(here::here("data", "processed", "measures.csv"),
              na = "NA", show_col_types = FALSE) |>
  mutate(
    speaker  = factor(speaker),
    item     = factor(item),
    # 対比コーディング: 交互作用を含むモデルでは処理コーディングの
    # 「主効果」は参照水準での単純効果になる。和対比にしておくと
    # 主効果を平均的な効果として解釈できる（第10章 §対比コーディング）。
    condition = factor(condition)
  )
contrasts(d$condition) <- contr.sum(nlevels(d$condition))

# 連続予測変数は中心化する。切片の解釈が可能になり、収束もしやすくなる。
d <- d |> mutate(logfreq_c = as.numeric(scale(log(frequency), scale = FALSE)))

# --- 1. maximal model から始める ---------------------------------------
# Barr et al. (2013) の「デザインで正当化される最大構造」。
# 収束しない場合に何を削るかは、恣意的に決めずに手順を決めておく（下記）。
m_max <- lmer(
  duration ~ condition * logfreq_c +
    (1 + condition | speaker) +
    (1 + condition | item),
  data = d, REML = TRUE,
  control = lmerControl(optimizer = "bobyqa",
                        optCtrl = list(maxfun = 2e5))
)

# --- 2. 収束しない場合の縮退手順（あらかじめ決めておく） ----------------
# (a) 相関パラメータを外す  (1 + condition || speaker)
# (b) 項目側のランダム傾きを外す
# (c) 話者側のランダム傾きを外す
# どこまで落としたかは必ず論文に書く。「収束しなかったので lm にした」は不可。
m_red <- lmer(
  duration ~ condition * logfreq_c +
    (1 + condition || speaker) +
    (1 | item),
  data = d, REML = TRUE, control = lmerControl(optimizer = "bobyqa")
)

# --- 3. 診断 ------------------------------------------------------------
print(summary(m_max))
cat("\n特異適合か: ", isSingular(m_max), "\n")

# 残差診断。正規性そのものより、系統的なパターンの有無を見る。
plot_resid <- tibble(fit = fitted(m_max), res = resid(m_max)) |>
  ggplot(aes(fit, res)) +
  geom_point(alpha = 0.25, size = 0.8) +
  geom_hline(yintercept = 0, linetype = 2) +
  geom_smooth(method = "loess", se = FALSE, linewidth = 0.6, formula = y ~ x) +
  labs(x = "予測値 (ms)", y = "残差 (ms)",
       caption = "scripts/R/10_mixed_models.R より生成")
save_fig(plot_resid, "fig10-02-residuals", chapter = "ch10")

# --- 4. 報告 ------------------------------------------------------------
# p 値だけを報告しない。推定値・標準誤差・信頼区間を併記する（style-guide §6）。
ci <- confint(m_max, method = "Wald", parm = "beta_")
print(cbind(Estimate = fixef(m_max), ci))


## `ch10_worked_example.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」通しの作例

「線形モデル・対比コーディング・交互作用」節の処理コーディング対比図と、
「混合効果モデル」節の残差診断プロットは、いずれも
data/processed/measures.csv (scripts/R/sim_measures.R が生成する架空データ)
を実際に解析した結果である。ここでは処理コーディングと和対比コーディングの
係数を並べる部分だけを担当する(残差診断はscripts/R/10_mixed_models.Rが担当)。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

d <- read_csv(here::here("data", "processed", "measures.csv"), show_col_types = FALSE) |>
  mutate(condition = factor(condition, levels = c("通常", "明瞭")))

d_treat <- d; contrasts(d_treat$condition) <- contr.treatment(2)
d_sum   <- d; contrasts(d_sum$condition)   <- contr.sum(2)

m_treat <- lm(duration ~ condition, data = d_treat)
m_sum   <- lm(duration ~ condition, data = d_sum)

coef_table <- tibble(
  coding = c("処理コーディング", "処理コーディング", "和対比コーディング", "和対比コーディング"),
  term   = c("(Intercept)", "condition2", "(Intercept)", "condition1"),
  estimate = c(coef(m_treat), coef(m_sum))
)
print(coef_table)

cat("\n通常条件の平均:", round(mean(d$duration[d$condition == "通常"]), 1), "ms\n")
cat("明瞭条件の平均:", round(mean(d$duration[d$condition == "明瞭"]), 1), "ms\n")
cat("全体平均:", round(mean(d$duration), 1), "ms\n")

means_df <- tibble(condition = c("通常", "明瞭"),
                    mean_dur = c(mean(d$duration[d$condition == "通常"]),
                                 mean(d$duration[d$condition == "明瞭"])))
grand_mean <- mean(d$duration)

# grand_mean(234.1)と通常条件の平均(223.1)は11msしか離れておらず、
# 各ラベルをその線の真上/真下に小さくずらすだけでは2行ラベル同士が
# 重なってしまう。そこで、和対比の切片ラベルは線より十分上、処理コー
# ディングの切片ラベルは線より十分下に離し、かつ棒の外側(左右)に
# 置くことで、線・棒・ラベルのどれとも重ならないようにする。
mean_normal <- means_df$mean_dur[means_df$condition == "通常"]

p <- ggplot(means_df, aes(condition, mean_dur)) +
  geom_col(width = 0.5, fill = "#34518A", alpha = 0.75) +
  geom_hline(yintercept = grand_mean, linetype = 2, colour = "#A8681F") +
  geom_hline(yintercept = mean_normal, linetype = 3, colour = "grey40") +
  annotate("text", x = 2.42, y = grand_mean + 22, label = "和対比の切片\n(全体平均)",
           hjust = 0, vjust = 0.5, size = 3, colour = "#A8681F", lineheight = 0.9) +
  annotate("text", x = 0.58, y = mean_normal - 22,
            label = "処理コーディングの切片\n(参照水準の平均)",
            hjust = 1, vjust = 0.5, size = 3, colour = "grey30", lineheight = 0.9) +
  coord_cartesian(ylim = c(0, 268), xlim = c(0.3, 2.7), clip = "off") +
  labs(x = NULL, y = "持続時間 (ms)",
       caption = "シミュレーションデータ。scripts/R/ch10_worked_example.R より生成。")
save_fig(p, "fig10-01-coding-comparison", chapter = "ch10", width = 6.5, height = 4.2)
cat("完了。図は assets/figures/ch10/ に保存した。\n")


## `ch10_model_comparison.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

「モデル比較と報告の作法」節の作例。交互作用の有無が異なる2つの
混合効果モデルを、最尤法(ML)で推定し、尤度比検定によって比較する。
scripts/R/sim_measures.R が生成する架空データ(measures.csv)を用いる。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

measures_path <- here::here("data", "processed", "measures.csv")
if (!file.exists(measures_path)) source(here::here("scripts", "R", "sim_measures.R"))
d <- read_csv(measures_path, show_col_types = FALSE) |>
  mutate(logfreq_c = as.numeric(scale(log(frequency), scale = FALSE)),
         condition = factor(condition))
contrasts(d$condition) <- contr.sum(2)

# 交互作用を含むモデルと含まないモデルを、いずれもML推定で当てはめる
m_full <- lmer(duration ~ condition * logfreq_c + (1 + condition | speaker) + (1 | item),
               data = d, REML = FALSE)
m_reduced <- lmer(duration ~ condition + logfreq_c + (1 + condition | speaker) + (1 | item),
                  data = d, REML = FALSE)

comparison <- anova(m_reduced, m_full)
print(comparison)

chisq <- comparison$Chisq[2]
df    <- comparison$Df[2]
p_val <- comparison$`Pr(>Chisq)`[2]
cat("\n尤度比検定: χ²(", df, ") =", round(chisq, 2), ", p =",
    format.pval(p_val, digits = 3), "\n")


## `ch10_effect_size.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

「効果量・区間・検定力」節の作例。条件間の平均差(ミリ秒)を、
標準化された効果量(Cohen's d)に変換する具体的な数値例。
scripts/R/sim_measures.R が生成する架空データを用いる。

このデータは話者20名×項目30語の交差デザインであり、各話者・各項目が
両条件に反復して現れる(対応のあるデータ)。単純に2条件をプールした
SDで割ると、話者間・項目間のばらつきまで「ノイズ」に含めてしまい、
対応のあるデザインの情報を無視した値になる(Westfall, Kenny, & Judd 2014)。
対応のある効果量(Cohen's d_z、話者×項目ペアごとの差のSDで標準化)を
あわせて計算し、両者の違いを示す。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

measures_path <- here::here("data", "processed", "measures.csv")
if (!file.exists(measures_path)) source(here::here("scripts", "R", "sim_measures.R"))
d <- read_csv(measures_path, show_col_types = FALSE)

by_cond <- d |> group_by(condition) |> summarize(mean_dur = mean(duration), sd_dur = sd(duration), n = n())
print(by_cond)

mean_diff <- diff(by_cond$mean_dur)
pooled_sd <- sqrt(mean(by_cond$sd_dur^2))
cohens_d_naive <- mean_diff / pooled_sd

cat("\n平均差:", round(mean_diff, 1), "ms\n")
cat("(参考、対応を無視した)プールされたSD:", round(pooled_sd, 1), "ms\n")
cat("(参考、対応を無視した)Cohen's d:", round(cohens_d_naive, 2), "\n\n")

wide <- d |>
  select(speaker, item, condition, duration) |>
  pivot_wider(names_from = condition, values_from = duration)
pair_diff <- wide[["明瞭"]] - wide[["通常"]]
cohens_dz <- mean(pair_diff) / sd(pair_diff)

cat("話者×項目ペアごとの差の平均:", round(mean(pair_diff), 1), "ms\n")
cat("差のSD:", round(sd(pair_diff), 1), "ms\n")
cat("対応のある効果量 Cohen's d_z:", round(cohens_dz, 2), "\n")
